# Stage 1: HURDLER compatibility by middle-module length

Every accepted unique middle module is evaluated against all eight plasmids with `legacy-optimized-v1`. The tables, not this notebook, contain all candidate solutions and the deterministic selected plasmid/RE pair.

In [ ]:
PER_MODULE = '/home/wendai/projects/hurdler/clone_repeat_protein/studies/hurdler_validation/step04_module_optimization/tables/expanded-middle-repeatsdb-foldseek-v1/module_compatibility.parquet'
BINNED = '/home/wendai/projects/hurdler/clone_repeat_protein/studies/hurdler_validation/step04_module_optimization/tables/expanded-middle-repeatsdb-foldseek-v1/module_compatibility_binned.parquet'
CANDIDATES = '/home/wendai/projects/hurdler/clone_repeat_protein/studies/hurdler_validation/step04_module_optimization/tables/expanded-middle-repeatsdb-foldseek-v1/module_compatibility_candidates.parquet'
FIGURE_STEM = '/home/wendai/projects/hurdler/clone_repeat_protein/studies/hurdler_validation/step04_module_optimization/figures/expanded-middle-repeatsdb-foldseek-v1/module_compatibility_by_length'

In [ ]:
from pathlib import Path
import hashlib, json
import pandas as pd
VERSION = 'expanded-middle-repeatsdb-foldseek-v1'
def sha256(path):
    path = Path(path)
    if not path.is_file(): return None
    h = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024*1024), b''): h.update(chunk)
    return h.hexdigest()
def read_optional(path):
    path = Path(path)
    if not path.is_file(): return None
    return pd.read_parquet(path) if path.suffix == '.parquet' else pd.read_csv(path)
run_context = {'corpus_version': VERSION, 'rules_version': 'legacy-optimized-v1', 'inputs': {}, 'row_counts': {}, 'filter_flow': [], 'limitations': [], 'status': 'passed'}

In [ ]:
from hurdler.module_experiments import plot_compatibility
import pyarrow.parquet as pq
per_module = read_optional(PER_MODULE); binned = read_optional(BINNED)
candidate_rows = pq.ParquetFile(CANDIDATES).metadata.num_rows if Path(CANDIDATES).is_file() else None
for path in (PER_MODULE,BINNED,CANDIDATES): run_context['inputs'][Path(path).name] = sha256(path)
if per_module is None or binned is None or candidate_rows is None:
    run_context['status']='production_pending'; run_context['limitations'].append('Compatibility shards have not yet been finalized; no provisional bar heights are shown.')
    overview=pd.DataFrame({'status':['production_pending']})
else:
    assert ~per_module.duplicated(['collection','unit_sequence']).any()
    assert (binned.compatible_count+binned.incompatible_count).eq(binned.total_count).all()
    assert per_module.hurdler_compatible.sum() == binned.compatible_count.sum()
    overview=per_module.groupby('collection').agg(modules=('module_id','size'),compatible=('hurdler_compatible','sum'),median_length_aa=('unit_length','median')).reset_index()
    overview['compatible_fraction']=overview.compatible/overview.modules
    plot_compatibility(binned, FIGURE_STEM)
run_context['row_counts']={'modules':None if per_module is None else len(per_module),'candidate_solutions':candidate_rows,'length_bins':None if binned is None else len(binned)}
overview

In [ ]:
binned if binned is not None else pd.DataFrame({'status':['production_pending']})

In [ ]:
from IPython.display import Image, display
figure=Path(str(FIGURE_STEM)+'.png')
if figure.is_file(): display(Image(filename=str(figure)))
else: print('Figure pending finalized Stage-1 tables:', figure)

In [ ]:
run_context['filter_flow']=['<6AA: repeat motif to shortest effective module >=6AA', 'scan effective module twice with frozen signed-overhang rules', 'evaluate all eight maintained plasmids', 'compatible iff at least one solution exists', 'retain all candidates and select by frozen deterministic ranking', 'shared 10-AA bins with empty intervening bins visible']
run_context['limitations'].append('Natural and Designed panels share bin order and compatibility encoding; corpus completeness depends on the reference notebook status.')

In [ ]:
run_context